In [1]:
!pip install selenium webdriver-manager beautifulsoup4


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [50]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
import urllib.parse
import time
from datetime import datetime

# 1. 브라우저 실행
driver = webdriver.Chrome()
driver.maximize_window()

df_data = []
scraped_base_urls = set()

keywords = [
    "밥태기", "빨래", "침구", "먼지", "땀띠", "스트레스", "청소", "분담", "체력", "가전", "우울", "번아웃", "통증", "수면", 
    "온도", "습도", "통잠", "수유", "밤수", "밤중 수유", "이유식", "분유", "태열", "정수기", 
    "에어컨", "가습기", "식기세척기", "청소기", "스피커", "등센서", "기갈대", "기저귀", 
    "육아템", "필수템", "육아"
]

try:
    # 2. 로그인
    driver.get("https://nid.naver.com/nidlogin.login")
    print("⚠️ 팝업된 브라우저에서 30초 내에 네이버 로그인을 진행해 주세요!")
    time.sleep(30)

    print(f"\n총 {len(keywords)}개의 키워드 크롤링을 시작합니다. (2021년 ~ 현재 글 무제한 수집)")
    print("-" * 50)

    for keyword in keywords:
        print(f"\n🔍 현재 검색 키워드: [{keyword}]")
        encoded_keyword = urllib.parse.quote(keyword)
        
        href_list = []
        page = 1
        stop_pagination = False # 2021년 이전 글을 만나면 페이지 넘김을 멈추기 위한 스위치
        
        # 4. 페이징 처리: 1페이지부터 끝까지(또는 2020년 글이 나올 때까지) 반복
        while not stop_pagination:
            target_url = f"https://cafe.naver.com/f-e/cafes/10298136/menus/46?viewType=L&ta=SUBJECT&q={encoded_keyword}&page={page}"
            driver.get(target_url)
            time.sleep(3) # 목록 로딩 대기
            
            rows = driver.find_elements(By.CSS_SELECTOR, 'tbody > tr')
            if not rows:
                print(f"  └ 더 이상 게시글이 없습니다. (페이지 {page}에서 탐색 종료)")
                break
                
            added_in_this_page = 0
            
            for row in rows:
                try:
                    tds = row.find_elements(By.TAG_NAME, 'td')
                    if len(tds) < 4:
                        continue
                    
                    date_text = tds[3].text.strip()
                    
                    # 연도 추출 (오늘 작성된 글은 '11:58' 처럼 시간이 뜨므로 올해로 간주)
                    if ':' in date_text:
                        year = datetime.now().year
                    else:
                        year = int(date_text.split('.')[0])
                        
                    # 2021년보다 과거의 글을 만나면? 탐색 완전 중단!
                    if year < 2021:
                        stop_pagination = True
                        break # 현재 페이지 탐색 중단
                        
                    # 2021년 이후의 글이라면 링크 수집 (3개 제한 없음!)
                    article_elem = row.find_element(By.CSS_SELECTOR, 'a.article')
                    href = article_elem.get_attribute('href')
                    
                    if href:
                        if not href.startswith('http'):
                            href = "https://cafe.naver.com" + href
                        
                        base_url = href.split('?')[0]
                        
                        if base_url not in scraped_base_urls:
                            scraped_base_urls.add(base_url)
                            href_list.append(href)
                            added_in_this_page += 1
                            
                except Exception as e:
                    continue

            print(f"  └ [페이지 {page}] {added_in_this_page}개의 링크 확보 (현재 누적: {len(href_list)}개)")
            
            if stop_pagination:
                print("  └ 🛑 2020년 이전 글이 발견되어 과거 글 탐색을 중단합니다.")
                break
                
            page += 1
            
            # 무한 루프 방지를 위한 안전장치 (최대 100페이지까지만)
            if page > 100:
                print("  └ 최대 탐색 가능 페이지(100)를 초과하여 탐색을 중단합니다.")
                break

        if not href_list:
            print(f"  └ ⚠️ [{keyword}] 2021년 이후에 작성된 새로운 글이 없습니다.")
            continue

        print(f"  └ 👉 수집된 총 {len(href_list)}개의 게시글 상세 정보 크롤링을 시작합니다...")

        # 5. 수집된 모든 상세 페이지 순회
        for idx, href in enumerate(href_list):
            driver.get(href)
            time.sleep(3) # 상세 페이지 로딩 대기
            
            try:
                driver.switch_to.frame('cafe_main')
                
                # (1) 제목
                try:
                    title = driver.find_element(By.CSS_SELECTOR, 'h3.title_text').text
                except:
                    title = "제목 없음"
                
                # (2) 작성자
                try:
                    author = driver.find_element(By.CSS_SELECTOR, '.nickname').text.strip()
                except:
                    author = "작성자 없음"

                # (3) 날짜 (상세페이지 본문 기준)
                try:
                    date = driver.find_element(By.CSS_SELECTOR, '.date').text
                except:
                    date = "날짜 정보 없음"
                    
                # (4) 본문
                try:
                    body = driver.find_element(By.CLASS_NAME, 'se-main-container').text
                except:
                    try:
                        body = driver.find_element(By.CSS_SELECTOR, '.se-component.se-text.se-l-default').text
                    except:
                        body = "본문을 불러올 수 없습니다."
                    
                # (5) 댓글
                comments_list = []
                try:
                    comment_items = driver.find_elements(By.CSS_SELECTOR, 'li.CommentItem')
                    for item in comment_items:
                        try:
                            c_writer = item.find_element(By.CSS_SELECTOR, '.comment_nickname').text.strip()
                            c_text = item.find_element(By.CSS_SELECTOR, '.text_comment').text.strip()
                            comments_list.append(f"[{c_writer}] {c_text}")
                        except:
                            pass
                    comments_str = "\n".join(comments_list) if comments_list else "댓글 없음"
                except:
                    comments_str = "댓글 없음"

                # 데이터 추가
                df_data.append([keyword, title, author, date, body, comments_str, href])
                print(f"    - [수집 성공 {idx+1}/{len(href_list)}] {title[:15]}...")
                
            except Exception as e:
                print(f"    - 게시글 수집 중 오류 발생: {e}")

    # 6. 최종 데이터프레임 저장
    cafe_df = pd.DataFrame(df_data, columns=['검색어', '제목', '작성자', '날짜', '본문', '댓글', '링크'])
    file_name = "레몬테라스_육아일기_전체수집_2021부터.csv"
    
    cafe_df.to_csv(file_name, encoding='utf-8-sig', index=False)
    
    print("\n" + "="*50)
    print(f"✅ 대량 크롤링이 완벽하게 끝났습니다! 파일: {file_name}")
    print(f"총 수집된 게시글 수: {len(cafe_df)}개")
    print("="*50)

except Exception as e:
    print(f"\n❌ 전체 실행 중 오류가 발생했습니다: {e}")

finally:
    driver.quit()
    print("\n✅ 웹 드라이버를 안전하게 종료했습니다.")

⚠️ 팝업된 브라우저에서 30초 내에 네이버 로그인을 진행해 주세요!

총 4개의 키워드 크롤링을 시작합니다. (2021년 ~ 현재 글 무제한 수집)
--------------------------------------------------

🔍 현재 검색 키워드: [밥태기]
  └ [페이지 1] 7개의 링크 확보 (현재 누적: 7개)
  └ 더 이상 게시글이 없습니다. (페이지 2에서 탐색 종료)
  └ 👉 수집된 총 7개의 게시글 상세 정보 크롤링을 시작합니다...
    - [수집 성공 1/7] 아기 밥태기 30개월 남아ㅠ...
    - [수집 성공 2/7] 9개월 밥태기...
    - [수집 성공 3/7] 후기인데 밥태기가 와버렸어요...
    - [수집 성공 4/7] 아이 밥태기 일까요? 너무 ...
    - [수집 성공 5/7] 밥을 너무안먹는아기 밥태기가...
    - [수집 성공 6/7] 10갤 밥태기..어쩌야될까요...
    - [수집 성공 7/7] 밥태기야 없어져라...

🔍 현재 검색 키워드: [빨래]
  └ [페이지 1] 3개의 링크 확보 (현재 누적: 3개)
  └ 🛑 2020년 이전 글이 발견되어 과거 글 탐색을 중단합니다.
  └ 👉 수집된 총 3개의 게시글 상세 정보 크롤링을 시작합니다...
    - [수집 성공 1/3] 일반 드럼세탁기로 아기빨래 ...
    - [수집 성공 2/3] 애기옷 빨래 쉰내..?ㅠ...
    - [수집 성공 3/3] 베란다빨래널고 커피수혈중~...

🔍 현재 검색 키워드: [침구]
  └ [페이지 1] 0개의 링크 확보 (현재 누적: 0개)
  └ 🛑 2020년 이전 글이 발견되어 과거 글 탐색을 중단합니다.
  └ ⚠️ [침구] 2021년 이후에 작성된 새로운 글이 없습니다.

🔍 현재 검색 키워드: [육아]
  └ [페이지 1] 15개의 링크 확보 (현재 누적: 15개)
  └ [페이지 2] 15개의 링크 확보 (현재 누적: 30개)
  └ [페이지 3] 15개의 링크 확보 (현재 누

KeyboardInterrupt: 

In [47]:
cafe_df

,검색어,제목,작성자,날짜,본문,댓글,링크
0,밥태기,아기 밥태기 30개월 남아ㅠㅠ,집에서뭐하뉘,2025.09.18. 11:43,원래도 밥을 잘 안먹는 아이인데 요즘 특히나 엄청나게 밥을 잘 안 먹어요. 밥 한 ...,[마나아라하하바자다] 2시간이라니... 진절머리 나네요ㅠㅠ 너무 힘드시겠어요 입맛에...,https://cafe.naver.com/f-e/cafes/10298136/arti...
1,밥태기,9개월 밥태기,핫트안아,2025.08.02. 23:03,이유식과 유아식 그 사이를 먹이고있는데\n잘먹던 밥전도 윀 뱉어버려요 휴\n아직 간...,댓글 없음,https://cafe.naver.com/f-e/cafes/10298136/arti...
2,먼지,미세먼지땜에 애 키우기가 힘들어요 에효,꾸꾸까가2,2025.03.24. 15:23,4월도 미세먼지는 아주아주 최악이내요\n애들 데리고 나갈라고 해도 비염에 콧물에 재...,[계단좀오를까] 으악...4월도 최악이내요\n차라리 그럼!!!! 집에서 노는 컨텐츠...,https://cafe.naver.com/f-e/cafes/10298136/arti...
3,먼지,미세먼지 왜이래요,레츠뤄언,2025.01.21. 06:58,코앞 건물도안보임...,[비오는날흙퍼먹기] 숨막혀요 ㅜ\n[샤걀의눈내리는마을] 중국인줄.\n정치도 그렇고 ...,https://cafe.naver.com/f-e/cafes/10298136/arti...
